# Spam Detection Web Application Using PyTorch and Flask

In this project, we build a complete end-to-end spam detection system using deep learning, PyTorch, and Flask. Instead of stopping after training a neural network, we take the next step toward real-world AI deployment by turning our trained model into an interactive web application.

The goal of this project is simple: given a text message or email, the model predicts whether the message is **Spam** or **Not Spam**.

This notebook demonstrates the full deployment workflow:

* Loading a previously trained PyTorch model
* Preprocessing user input text
* Running predictions with the neural network
* Building a simple web interface using Flask
* Allowing users to enter messages directly from a browser
* Returning real-time spam classification results

The project also introduces important production concepts such as:

* Model persistence using `torch.save()` and `load_state_dict()`
* Inference mode using `model.eval()`
* Feature preprocessing consistency
* Web APIs and browser interaction
* Deploying AI systems through Flask applications

By the end of this notebook, you will have a small production-style AI application that users can access through a web browser and interact with just like a real-world machine learning service.


# Step 1: Install Flask
We install Flask to create a small web application inside Google Colab.

In [1]:
!pip install flask

# Step 2: Import Required Libraries
These libraries are used for Flask, PyTorch, threading, and generating browser-accessible URLs in Colab.

In [2]:
from flask import Flask, request

import torch

import torch.nn as nn

import threading

from google.colab.output import eval_js

# Step 3: Define the SAME Neural Network Architecture
This architecture must exactly match the architecture used during training.

In [3]:
class SpamClassifier(nn.Module):

    def __init__(self):

        super(SpamClassifier, self).__init__()

        self.fc1 = nn.Linear(4, 16)

        self.fc2 = nn.Linear(16, 8)

        self.fc3 = nn.Linear(8, 1)

        self.relu = nn.ReLU()

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        x = self.relu(self.fc1(x))

        x = self.relu(self.fc2(x))

        x = self.sigmoid(self.fc3(x))

        return x

# Step 4: Load the Trained Model
We load the previously trained model weights from spam_model.pth.

In [4]:
model = SpamClassifier()

model.load_state_dict(

    torch.load(
        "spam_model.pth",
        map_location=torch.device("cpu")
    )
)

model.eval()

print("✅ Model loaded successfully!")

✅ Model loaded successfully!


# Step 5: Create Preprocessing Function
This function converts text into a simple numerical tensor representation.

In [5]:
def preprocess(text):

    text = text.lower()

    vector = torch.zeros(1, 4)

    spam_words = [
        "free",
        "win",
        "money",
        "offer"
    ]

    for i, word in enumerate(spam_words):

        if word in text:

            vector[0][i] = 1

    return vector

# Step 6: Create Flask App
Initialize the Flask web application.

In [6]:
app = Flask(__name__)

# Step 7: Create Homepage Route
This route displays the webpage interface.

In [7]:
@app.route("/")

def home():

    return """

    <html>

    <head>

        <title>Spam Detection App</title>

        <style>

            body {

                font-family: Arial;

                background-color: #f4f4f4;

                margin: 40px;
            }

            .container {

                background: white;

                padding: 30px;

                border-radius: 10px;

                max-width: 600px;

                margin: auto;
            }

            textarea {

                width: 100%;

                height: 120px;

                padding: 10px;

                font-size: 16px;
            }

            button {

                margin-top: 15px;

                padding: 12px 20px;

                font-size: 16px;

                cursor: pointer;
            }

        </style>

    </head>

    <body>

        <div class="container">

            <h1>📧 Spam Detection Web App</h1>

            <form action="/predict" method="post">

                <textarea
                    name="text"
                    placeholder="Enter your message here..."
                ></textarea>

                <br>

                <button type="submit">

                    Predict

                </button>

            </form>

        </div>

    </body>

    </html>

    """


# Step 8: Create Prediction Route
This route receives user input and returns the prediction result.

In [8]:
@app.route("/predict", methods=["POST"])

def predict():

    try:

        text = request.form["text"]

        input_tensor = preprocess(text)

        input_tensor = input_tensor.float()

        print("Input Tensor:")
        print(input_tensor)

        print("Input Shape:")
        print(input_tensor.shape)

        with torch.no_grad():

            output = model(input_tensor)

            print("Raw Output:")
            print(output)

            probability = output.squeeze().item()

        if probability >= 0.5:

            result = "🚨 Spam Detected!"

        else:

            result = "✅ Not Spam"

        return f"""

        <html>

        <head>

            <title>Prediction Result</title>

        </head>

        <body style="font-family:Arial;margin:40px;">

            <h1>📬 Prediction Result</h1>

            <p><b>Message:</b></p>

            <p>{text}</p>

            <h2>{result}</h2>

            <p>

                <b>Spam Probability:</b>
                {probability:.4f}

            </p>

            <a href="/">

                ← Try Another Message

            </a>

        </body>

        </html>

        """

    except Exception as e:

        return f"""

        <h1>ERROR</h1>

        <pre>{str(e)}</pre>

        """

# Step 9: Run Flask Server
Run Flask in a background thread inside Google Colab.

In [9]:
PORT = 5001

def run():

    print("🚀 Starting Flask server...")

    app.run(

        host="0.0.0.0",

        port=PORT,

        debug=False,

        use_reloader=False
    )

thread = threading.Thread(target=run)

thread.daemon = True

thread.start()

print(f"✅ Flask server running on port {PORT}")

🚀 Starting Flask server...
✅ Flask server running on port 5001


# Step 10: Generate Browser URL
Generate a public browser-accessible URL inside Google Colab.

In [10]:
public_url = eval_js(

    f"google.colab.kernel.proxyPort({PORT})"
)

print("\n🌍 Open this URL in your browser:\n")

print(public_url)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5001
 * Running on http://172.28.0.12:5001
INFO:werkzeug:Press CTRL+C to quit



🌍 Open this URL in your browser:

https://5001-m-s-kkb-usw1c0-c8ahz5ekjvem-c.us-west1-0.prod.colab.dev
